# Summit 2026 HOL — MCP Connectors in Cortex Agents
## Module 05: Connect Your Agent to Enterprise Systems

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🔹 Explains what **MCP (Model Context Protocol)** is and why it matters
2. 🛠️ Connects the Nexus Capital agent to **Atlassian (Jira/Confluence)** — works on any Snowflake account
3. 🛠️ Shows how to add the connector to the agent and authenticate
4. 🛠️ Demonstrates the agent **taking action** — creating tickets, searching Confluence
5. 📌 (Optional) Provides full setup instructions for **Google Workspace** and **Slack** on demo accounts

---

### What is MCP?

**Model Context Protocol (MCP)** is an open standard that lets AI agents securely interact with external business applications. Think of it as a universal adapter between your AI agent and tools like Jira, Salesforce, Slack, GitHub, etc.

Without MCP, your agent can only *answer questions*. With MCP, your agent can:
- Create a Jira ticket when it finds a compliance issue
- Pull a Salesforce client record to enrich its answer
- Post an alert to Slack when a risk threshold is breached
- Create a Confluence page to document its findings

**This transforms the agent from a Q&A tool into an enterprise automation platform.**

---

### How MCP Connectors Work in Snowflake:

```
┌──────────────────────┐          ┌─────────────────────┐
│  Snowflake Agent     │          │  External Service   │
│                      │  OAuth   │  (Jira, Salesforce) │
│  ┌────────────────┐  │◄────────►│                     │
│  │ External MCP   │  │  tools/  │  MCP Server         │
│  │ Server Object  │  │  list &  │  (hosted by vendor) │
│  └────────────────┘  │  call    │                     │
│         │            │          └─────────────────────┘
│         ▼            │
│  ┌────────────────┐  │
│  │ API Integration │ │
│  │ (OAuth creds)   │ │
│  └────────────────┘  │
└──────────────────────┘
```

1. **API Integration** — Stores OAuth credentials for the external service
2. **External MCP Server** — References the API integration and the vendor's MCP endpoint
3. **Agent Configuration** — The MCP server is added to the agent's spec
4. **User Authentication** — Each SI user connects their own identity via OAuth

> **Documentation:** [MCP Connectors](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp-connectors)

 ### 🔹 Available MCP Connectors:

| Connector | Auth Type | Setup | Notes |
|-----------|-----------|-------|-------|
| **Atlassian** (Jira/Confluence) | Dynamic Client Registration | Easiest — no manual OAuth app | Works on any account |
| **GitHub** | Standard OAuth | Create GitHub App | Works on any account |
| **Glean** | Dynamic Client Registration | Copy server URL | Works on any account |
| **Linear** | Dynamic Client Registration | Easy | Works on any account |
| **Salesforce** | Standard OAuth | Create Connected App | Works on any account |
| **Gmail** | Standard OAuth | Create Google Cloud OAuth app | Requires security approval for demo accounts, or use a personal Google account |
| **Google Drive** | Standard OAuth | Same Google Cloud app | Requires security approval for demo accounts, or use a personal Google account |
| **Google Calendar** | Standard OAuth | Same Google Cloud app | Requires security approval for demo accounts, or use a personal Google account |
| **Slack** | Standard OAuth | Create Slack app | Requires security approval for demo accounts, or use a personal Slack workspace |
| **Custom** | OAuth2 (manual) | Any MCP endpoint | Flexible |

> **On Snowhouse:** If you have access to Snowhouse, Gmail, Google Drive, Google Calendar, and Slack are available via **Browse 
Connectors** (one-click setup, no OAuth app creation). This lab uses a demo account, so we'll set up Atlassian (zero config) and provide
optional instructions for Google/Slack.

---

### Estimated Time: **20–25 minutes** (core) + 15 min (optional Google setup)

### Prerequisites:
- Module 04 complete (NEXUS_AGENT accessible in SI)
- For Atlassian: An Atlassian account (free tier works)
- For Google (optional): A personal Gmail account + Google Cloud Console access

## Step 1: Set Context

In [ ]:
%%sql -r SetContext
USE ROLE ACCOUNTADMIN;
USE DATABASE NEXUS_HOL;
USE SCHEMA AGENTS;
USE WAREHOUSE NEXUS_WH;

In [ ]:
%%sql -r GrantPriviledges
-- Grant privilege so NEXUS_ADMIN can create MCP servers in this schema
GRANT CREATE EXTERNAL MCP SERVER ON SCHEMA NEXUS_HOL.AGENTS TO ROLE NEXUS_ADMIN;

## Step 2: 🛠️ Create the Atlassian MCP Connector

### Three objects are required to connect an external MCP service:

| # | Object | What It Does |
|---|--------|-------------|
| 1 | **API Integration** | Stores OAuth configuration (how to authenticate with the external service) |
| 2 | **External MCP Server** | References the integration + the vendor's MCP endpoint URL |
| 3 | **GRANT** | Gives your agent's role permission to use the MCP server |

Atlassian uses **Dynamic Client Registration (DCR)** — Snowflake handles OAuth app registration automatically. No client IDs, no secrets, no manual app creation.

**Pre-requisite on Atlassian's side:**

Your Atlassian admin must add callback URLs at: **admin.atlassian.com** → Apps → AI Settings → Rovo MCP Server → Your domains:
- `https://apps-api.c1.us-west-2.aws.app.snowflake.com/oauth/complete-secret`
- `https://<your-account>.snowflakecomputing.com/oauth/complete-secret`

> **Tip:** Get your account URL from: 
> ```sql
> SELECT CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME()
>      || '.snowflakecomputing.com' AS account_url;
> ```

> 📌 **Note:** `OR REPLACE` is currently not supported for `external_mcp` API integrations.
> If you rerun this step and the integration already exists, either:
>
> ```sql
> DROP API INTEGRATION nexus_atlassian_integration;
> ```
>
> and rerun the cell, or create the integration with a different name.

In [ ]:
-- Create API Integration (DCR — no client ID/secret needed)
CREATE API INTEGRATION nexus_atlassian_integration
  API_PROVIDER = external_mcp
  API_ALLOWED_PREFIXES = ('https://mcp.atlassian.com')
  API_USER_AUTHENTICATION = (
    TYPE = OAUTH_DYNAMIC_CLIENT,
    OAUTH_RESOURCE_URL = 'https://mcp.atlassian.com/v1/mcp'
  )
  ENABLED = TRUE
  COMMENT = 'Nexus HOL - Atlassian MCP connector (Jira + Confluence)';

SELECT 'API Integration created' AS STATUS;

In [ ]:
%%sql -r CreateMCPServer
-- Create External MCP Server referencing the integration
-- ╔══════════════════════════════════════════════════════════════════════════════════════╗
-- ║  XXX: Replace with Atlassian's MCP endpoint URL.                                     ║
-- ║  Hint: It matches the OAUTH_RESOURCE_URL you used in the API Integration above.      ║
-- ╚══════════════════════════════════════════════════════════════════════════════════════╝
CREATE OR REPLACE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_ATLASSIAN_MCP
  WITH DISPLAY_NAME = 'Atlassian (Jira & Confluence)'
  URL = 'XXX'
  API_INTEGRATION = nexus_atlassian_integration
  COMMENT = 'Nexus HOL - Atlassian MCP server for creating Jira issues and searching Confluence';

SELECT 'Atlassian External MCP Server created' AS STATUS;

In [ ]:
%%sql -r GrantMCPUsage
-- Grant USAGE to NEXUS_ADMIN so the agent can access the MCP server
    GRANT USAGE ON EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_ATLASSIAN_MCP TO ROLE NEXUS_ADMIN;

    SELECT 'USAGE granted to NEXUS_ADMIN' AS STATUS;

## Step 3: 🛠️ Add the MCP Connector to the Agent

### Wire the Atlassian MCP server into NEXUS_AGENT's specification

This tells the agent it can discover and invoke tools from Atlassian (create Jira issues, search Confluence, comment on tickets).

> **Important:** `ALTER AGENT MODIFY LIVE VERSION SET SPECIFICATION` **replaces** the entire spec. You must include all existing tools + the new `mcp_servers` block.

In [ ]:
%%sql -r AddMCPConnector
USE ROLE NEXUS_ADMIN;

ALTER AGENT NEXUS_HOL.AGENTS.NEXUS_AGENT
  MODIFY LIVE VERSION SET SPECIFICATION =
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 45
    tokens: 16000

instructions:
  response: |
    You are the Nexus Capital AI Analyst. You help portfolio managers, relationship managers,
    and compliance officers understand client portfolios, trading activity, and market research.
    
    Guidelines:
    - Be concise and data-driven. Lead with numbers when available.
    - When showing financial data, format large numbers (e.g., $2.5B not $2500000000).
    - If a question spans both structured data and unstructured data, use BOTH tools.
    - Always cite which data source your answer came from.
    - For compliance questions, flag any potential issues clearly.
    - When a user asks to create a ticket or log an issue, use the Atlassian connector.

  orchestration: |
    - For client AUM, portfolio values, positions, or trades: use the Analyst tool.
    - For market outlook, research opinions, or compliance reports: use the Search tool.
    - For creating Jira tickets or searching Confluence: use the Atlassian MCP connector.

  sample_questions:
    - question: "Who are our top 5 clients by AUM?"
    - question: "Are there any compliance concerns? Create a Jira ticket if so."
    - question: "What is our tech sector exposure?"
    - question: "Search Confluence for our risk management policy."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "nexus_analytics"
      description: "Query structured financial data including client accounts, portfolio positions, trade history, and business metrics."
  - tool_spec:
      type: "cortex_search"
      name: "nexus_research"
      description: "Search analyst research notes, market commentary, risk assessments, and compliance reports."
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generate visualizations from query results."

tool_resources:
  nexus_analytics:
    semantic_view: "NEXUS_HOL.SEMANTIC.NEXUS_CAPITAL_SV"
    execution_environment:
      type: warehouse
      warehouse: "NEXUS_WH"
  nexus_research:
    name: "NEXUS_HOL.AGENTS.NEXUS_RESEARCH_SEARCH"
    max_results: "5"

mcp_servers:
  - server_spec:
      name: "NEXUS_HOL.AGENTS.NEXUS_ATLASSIAN_MCP"
$$;

SELECT 'Agent updated with Atlassian MCP connector' AS STATUS;

## Step 4: 🛠️ Authenticate and Test

### Connect to Atlassian

You can authenticate from either the **Agent Admin page** or the **CoWork chat**:

**Option A: From Agent Admin**
1. Go to **AI & ML → Cortex AI → Agents → NEXUS_AGENT**
2. Click the **MCP Connectors** tab
3. Click **Connect** next to Atlassian (Jira & Confluence)

**Option B: From CoWork chat**
1. Open **CoWork (formerly Snowflake Intelligence)** → select **Nexus Capital Analyst**
2. Click the **+** menu → **@ Connectors**
3. Click **Connect** next to Atlassian

**Both paths lead to the same consent screen:**
- It shows the Snowflake client name requesting access
- Verify that **Jira**, **Confluence**, and **Compass** are all checked
- Click **Approve**
- Sign in with your **@snowflake.com email** (this connects to your org's Jira/Confluence workspace)
- Authorize access → you're connected!

---

### Demo — Agent Takes Action:

Go to **CoWork** and try:

> *"Are there any compliance concerns with our client portfolios? If you find any, create a Jira ticket to track it."*

The agent will:
1. Search research notes for compliance flags (Cortex Search)
2. Find the Velocity Capital TSLA concentration warning
3. Create a Jira issue with the details (Atlassian MCP)

---

### Other things to try:
- "Search Confluence for our investment policy"
- "Create a Jira ticket titled 'Review TSLA concentration for Velocity Capital' with priority High"
- "What Jira tickets are assigned to me?"

> **Note:** Depending on your Atlassian permissions, some actions may be read-only. If Jira ticket creation fails or you can't see assigned tickets, this is expected — the integration may only have Confluence scopes authorized. For this lab, the important thing is that the agent **attempts** the action and explains what it needs. In a customer deployment, the Atlassian admin would grant full Jira read/write scopes during the initial connector setup.

## Step 5: ✅ Verify Configuration

In [ ]:
%%sql -r ShowMCPServers
-- List all External MCP Servers in our schema
SHOW EXTERNAL MCP SERVERS IN SCHEMA NEXUS_HOL.AGENTS;

In [ ]:
%%sql -r DescribeMCPServers
-- Describe the Atlassian MCP server
DESCRIBE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_ATLASSIAN_MCP;

## Step 6: 🔹 Security and Governance

### MCP Security Model:

| Concern | How Snowflake Handles It |
|---------|------------------------|
| Who can connect? | USAGE privilege on the MCP server object — RBAC controlled |
| What data leaves Snowflake? | Only the agent's tool call request (e.g., "create ticket with title X") — not raw table data |
| How are credentials stored? | OAuth tokens managed by Snowflake, per-user, auto-rotating |
| Audit trail? | All MCP tool calls logged in Account Usage |
| Disable access? | `ALTER API INTEGRATION ... SET ENABLED = FALSE` — instantly revokes all tokens |

> **Documentation:** [MCP Connectors — Key Considerations](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp-connectors#key-considerations-and-best-practices)

---

## 📌 Optional: Google Workspace Setup (Gmail, Drive, Calendar)

### For SEs who want Google connectors on their demo account — or need to guide customers through the setup

> **Quick path (Snowhouse users):** If you have access to Snowhouse, Gmail, Google Drive, Google Calendar, and Slack are available via **Browse Connectors** (AI & ML → Cortex AI → Agents → Settings → Tools and Connectors → Browse Connectors). One-click setup, no OAuth app creation needed.

> **Demo accounts:** Google MCP connectors require you to create a Google Cloud OAuth app with a **personal Google account** (not @snowflake.com). Snowflake's corporate Google Workspace blocks third-party OAuth apps on demo accounts.

---

### Part A: Create a Google Cloud OAuth App

1. Go to [console.cloud.google.com](https://console.cloud.google.com) — sign in with your **personal Gmail**
2. Create a **New Project** (e.g., "Nexus HOL MCP") — no billing/free trial needed
3. Navigate to **Google Auth Platform** (search "OAuth" in the top search bar)
4. Click **Get Started** (blue button in the center of the page)
5. Select **External** user type → complete the consent screen setup
6. Navigate to **Clients** (left nav) → **+ Create Client**
7. Application type: **Web application**
8. Name: `Snowflake MCP`
9. Add **Authorized redirect URIs**:
   - `https://apps-api.c1.us-west-2.aws.app.snowflake.com/oauth/complete-secret`
   - `https://<your-account>.snowflakecomputing.com/oauth/complete-secret`
10. Click **Create** — copy the **Client ID** and **Client Secret** (shown only once)
11. Go to **APIs & Services → Library** and enable:
    - Gmail API
    - Google Drive API
    - Google Calendar API

> **Note:** You do NOT need to sign up for Google Cloud billing or a free trial. The free tier covers OAuth app creation.

> **To find your redirect URIs:** Run `SELECT SYSTEM$ALLOWLIST();` in Snowflake. Use the `SNOWSIGHT_DEPLOYMENT` entry starting with `apps-api` and your `SNOWFLAKE_DEPLOYMENT` host.

### 🔌 Part B: Create the Google Connectors in Snowflake

Replace `<your-client-id>` and `<your-client-secret>` with the values from Google Cloud Console:

In [ ]:
-- OPTIONAL: Google Workspace MCP Connectors
-- Replace <your-client-id> and <your-client-secret> with your Google Cloud OAuth credentials

USE ROLE ACCOUNTADMIN;

-- Create API Integration for Google Workspace
CREATE API INTEGRATION nexus_google_integration
  API_PROVIDER = external_mcp
  API_ALLOWED_PREFIXES = ('https://gmail.mcp.snowflake.com', 'https://gdrive.mcp.snowflake.com', 'https://gcal.mcp.snowflake.com')
  API_USER_AUTHENTICATION = (
    TYPE = OAUTH2,
    OAUTH_CLIENT_ID = '<your-client-id>',
    OAUTH_CLIENT_SECRET = '<your-client-secret>',
    OAUTH_TOKEN_ENDPOINT = 'https://oauth2.googleapis.com/token',
    OAUTH_AUTHORIZATION_ENDPOINT = 'https://accounts.google.com/o/oauth2/v2/auth',
    OAUTH_CLIENT_AUTH_METHOD = CLIENT_SECRET_BASIC,
    OAUTH_REFRESH_TOKEN_VALIDITY = 86400
  )
  ENABLED = TRUE
  COMMENT = 'Nexus HOL - Google Workspace MCP connector';

-- Create Gmail MCP Server
CREATE OR REPLACE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GMAIL_MCP
  WITH DISPLAY_NAME = 'Gmail'
  URL = 'https://gmail.mcp.snowflake.com/mcp'
  API_INTEGRATION = nexus_google_integration;

-- Create Google Drive MCP Server
CREATE OR REPLACE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GDRIVE_MCP
  WITH DISPLAY_NAME = 'Google Drive'
  URL = 'https://gdrive.mcp.snowflake.com/mcp'
  API_INTEGRATION = nexus_google_integration;

-- Create Google Calendar MCP Server
CREATE OR REPLACE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GCAL_MCP
  WITH DISPLAY_NAME = 'Google Calendar'
  URL = 'https://gcal.mcp.snowflake.com/mcp'
  API_INTEGRATION = nexus_google_integration;

-- Grant access to NEXUS_ADMIN
GRANT USAGE ON EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GMAIL_MCP TO ROLE NEXUS_ADMIN;
GRANT USAGE ON EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GDRIVE_MCP TO ROLE NEXUS_ADMIN;
GRANT USAGE ON EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_GCAL_MCP TO ROLE NEXUS_ADMIN;

SELECT 'Google Workspace MCP Servers created (Gmail, Drive, Calendar)' AS STATUS;

### 💬 Part C: Slack (Same Pattern)

Slack follows the same custom OAuth approach as Google. You'd need to:
1. Create a Slack App at [api.slack.com/apps](https://api.slack.com/apps)
2. Add OAuth scopes (`search:read`, `chat:write`, `channels:history`, `users:read`, etc.)
3. Set redirect URLs (same two URLs as Google above)
4. Create the integration and MCP server:

```sql
-- Slack MCP Server (replace with your Slack app credentials)
CREATE API INTEGRATION nexus_slack_integration
  API_PROVIDER = external_mcp
  API_ALLOWED_PREFIXES = ('https://mcp.slack.com')
  API_USER_AUTHENTICATION = (
    TYPE = OAUTH2,
    OAUTH_CLIENT_ID = '<your-slack-client-id>',
    OAUTH_CLIENT_SECRET = '<your-slack-client-secret>',
    OAUTH_TOKEN_ENDPOINT = 'https://slack.com/api/oauth.v2.access',
    OAUTH_AUTHORIZATION_ENDPOINT = 'https://slack.com/oauth/v2/authorize',
    OAUTH_CLIENT_AUTH_METHOD = CLIENT_SECRET_BASIC,
    OAUTH_REFRESH_TOKEN_VALIDITY = 86400
  )
  ENABLED = TRUE;

CREATE OR REPLACE EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_SLACK_MCP
  WITH DISPLAY_NAME = 'Slack'
  URL = 'https://mcp.slack.com/mcp'
  API_INTEGRATION = nexus_slack_integration;

GRANT USAGE ON EXTERNAL MCP SERVER NEXUS_HOL.AGENTS.NEXUS_SLACK_MCP TO ROLE NEXUS_ADMIN;
```

> **On Snowhouse:** Slack is available via Browse Connectors with one-click setup — no app creation needed.

### 📌 MCP Server URLs (Reference)

These are **Snowflake-hosted proxy endpoints** — Snowflake handles the MCP protocol translation and OAuth token management.

| Connector | Snowflake-Hosted MCP URL | Auth Type |
|-----------|-------------------------|----------|
| Atlassian | `https://mcp.atlassian.com/v1/mcp` | DCR (no credentials needed) |
| Gmail | `https://gmail.mcp.snowflake.com/mcp` | OAuth2 (client ID + secret) |
| Google Drive | `https://gdrive.mcp.snowflake.com/mcp` | OAuth2 (client ID + secret) |
| Google Calendar | `https://gcal.mcp.snowflake.com/mcp` | OAuth2 (client ID + secret) |
| Slack | `https://mcp.slack.com/mcp` | OAuth2 (client ID + secret) |

---

## ✅ Module 05 Complete!

### You now have:
- **Atlassian MCP connector** configured and added to NEXUS_AGENT
- Agent can create Jira issues, search Confluence, comment on tickets
- Optional: Google Workspace connectors (Gmail, Drive, Calendar) with full setup instructions
- Reference: Slack connector setup (same pattern)
- Per-user OAuth authentication for governed external access

---

### Key Takeaways:

1. **MCP transforms agents from answering machines into action-takers.** The agent doesn't just tell you about a compliance issue — it creates a ticket to track it.

2. **Two setup paths:**
   - **DCR connectors** (Atlassian, Glean, Linear) — 3 SQL statements, no OAuth app needed
   - **OAuth2 connectors** (Google, Slack, Salesforce) — create an OAuth app on the provider side, then 3 SQL statements

3. **Security is per-user.** Each person authenticates with their own identity. No shared service accounts. Full audit trail.

4. **Snowhouse shortcut.** If your account has Browse Connectors, Google/Slack/Atlassian are one-click setup.

---

### Talking Points:

- "MCP is to AI agents what APIs were to web apps — a universal integration standard."
- "Your agent can now CREATE a Jira ticket, not just tell you about a problem."
- "Every external action is OAuth-scoped to the individual user — no shared credentials, full audit trail."
- "Snowflake hosts the MCP proxy — your data never goes directly to the third-party service."

---

### Next Steps:

Continue to **Module 06: AI Security & Governance** — Configure agent identity, credit budgets, trust boundaries, and RBAC policies.